In [38]:
import torch
from torch import nn
import torch.nn.functional as F
device = ('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
from einops import rearrange, repeat
torch.manual_seed(2023)
from fla.modules import FusedRMSNormSwishGate,  ShortConvolution
try:
    from fla.modules.l2norm import l2_norm as l2_norm_fn
except:
    from fla.modules.l2norm import l2_norm_fn
from package.chunk import chunk_gated_delta_rule
class MLP(nn.Module):
    def __init__(self,input_dim,hidden_dim,output_dim):
        super(MLP,self).__init__()
        self.fc1 = nn.Linear(input_dim,hidden_dim)
        self.fc2 = nn.Linear(hidden_dim,output_dim)
        self.relu = nn.ReLU()

    def forward(self,x): 
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x
    

class Patchembed(nn.Module):
    def __init__(self,in_dim = 1,out_dim = 64,patch_size = 2,img_size = 28):
        super(Patchembed,self).__init__()
        self.proj = nn.Conv2d(in_dim,out_dim,patch_size,patch_size,bias = False)
        self.num_patches = (img_size//patch_size)**2
        self.pos = nn.Parameter(torch.randn(1,self.num_patches,out_dim))

    def forward(self,x):
        x = self.proj(x)
        x = x.flatten(2).transpose(1,2)
        x = x + self.pos
        return x
    
class RMSNorm(torch.nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim,dtype = torch.float32))

    def _norm(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)

    def forward(self, x):
        output = self._norm(x.float()).type_as(x)
        return output * self.weight

class Conv1dProj(nn.Module):
    def __init__(self,dim,kernel_size):
        super(Conv1dProj,self).__init__()
        self.conv = nn.Conv1d(dim,dim,kernel_size,padding = kernel_size//2,groups = dim)
    
    def forward(self,x):
        return self.conv(x.transpose(1,2)).transpose(1,2)



cuda


In [ ]:
class GDR_Chunkwise(nn.Module):
    def __init__(self,
                 hidden_size = 128,
                 expand_k = 0.75,
                 expand_v = 1.5,
                 num_heads = 8,
                 num_kv_heads = None,
                 conv_size = 4,
                 eps = 1e-6,
                 elementwise_affine = True,
                 ):
        super(GDR_Chunkwise,self).__init__()
        self.hidden_size = hidden_size
        self.expand_k = expand_k
        self.expand_v = expand_v
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads if num_kv_heads is not None else num_heads
        self.num_kv_group = self.num_heads //self.num_kv_heads
        self.conv_size = conv_size
        self.eps = eps
        self.key_dim = int(hidden_size*expand_k)
        self.value_dim = int(hidden_size*expand_v)
        self.key_dim_per_group = self.key_dim // self.num_kv_group
        self.value_dim_per_group = self.value_dim // self.num_kv_group
        self.head_qk_dim = self.key_dim//num_heads
        self.head_v_dim = self.value_dim//num_heads
        self.q_proj = nn.Linear(hidden_size,self.key_dim,bias = False)
        self.k_proj = nn.Linear(hidden_size,self.key_dim,bias = False)
        self.v_proj = nn.Linear(hidden_size,self.value_dim_per_group,bias = False)
        self.gate_proj = nn.Linear(hidden_size,self.num_heads,bias = False)
        self.q_conv = ShortConvolution(self.key_dim, conv_size, activation='silu')
        self.k_conv = ShortConvolution(self.key_dim_per_group, conv_size, activation='silu')
        self.v_conv = ShortConvolution(self.value_dim_per_group, conv_size, activation='silu')
        self.act = nn.SiLU()
        self.gk_proj = nn.Linear(hidden_size,self.num_heads,bias = False)
        self.b_proj = nn.Linear(hidden_size,self.num_heads,bias = False)
        self.o_proj = nn.Linear(self.value_dim,hidden_size,bias = False)
        self.residual_proj = nn.Linear(hidden_size,hidden_size,bias = False)
        self.norm = FusedRMSNormSwishGate(self.head_v_dim, elementwise_affine,eps)
        self.D = nn.Parameter(torch.ones(self.num_heads))
        self.D._no_weight_decay = True

    def forward(self,x):
        last_state = None
        conv_state_q = None
        conv_state_k = None
        conv_state_v = None
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)
        q,_ = self.q_conv(q)
        k,_ = self.k_conv(k)
        v,_ = self.v_conv(v)
        gk = self.gk_proj(x).float()
        gk = F.logsigmoid(gk)/16  #对于源代码的gate_logit_normalizer  用来缩放门控值
        gk = gk.transpose(1,2)
        beta = self.b_proj(x).float().sigmoid()
        beta = beta.transpose(1,2)
        q = rearrange(q,'b l (h d) -> b h l d',h = self.num_heads)
        if self.num_kv_group > 1:
            k = repeat(k,'b l (h d) -> b (h g) l d',h = self.num_kv_heads,g = self.num_kv_group)
            v = repeat(v,'b l (h d) -> b (h g) l d',h = self.num_kv_heads,g = self.num_kv_group)
        else:
            k = rearrange(k,'b l (h d) -> b h l d',h = self.num_kv_heads)
            v = rearrange(v,'b l (h d) -> b h l d',h = self.num_kv_heads)
        q = l2_norm_fn(q).to(v)
        k = l2_norm_fn(k).to(v)
        recurrent_state = None
        o,recurrent_state = chunk_gated_delta_rule(q,k,v,beta,gk,initial_state = recurrent_state,output_final_state = False)
        o = o + self.D[None,:,None,None]*v
        o = rearrange(o,'b h l d -> b l h d')
        g = self.gate_proj(x)
        g = rearrange(g,'b l (h d) -> b l h d',h = self.num_heads)
        o = self.norm(o,g)
        o = rearrange(o,'b l h d -> b l (h d)')
        return o


        


In [45]:
x = x = torch.randn(9, 9, 128).to(device)
print(x.shape)
model = GDR_Chunkwise().to(device)
y = model(x)
print(y.shape)

torch.Size([9, 9, 128])
1
2
3
4
5
6
7
torch.Size([9, 9, 192])
